In [19]:
import warnings
warnings.filterwarnings("ignore")


In [20]:

!pip install ddgs

In [21]:
!pip install pydub openai sounddevice scipy transformers torch accelerate librosa
!apt-get install ffmpeg -y

E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?


In [22]:
system_prompt = """
You are a voice assistant.

If the user's question depends on current, real-time, or changing information
such as prices, weather, news, sports scores, or recent events,
you MUST call the web_search tool.

If the question is general knowledge or conceptual, answer directly.

Do not guess real-time information.
"""

In [ ]:
import os
from openai import OpenAI
import json

# ========== SET YOUR API KEYS HERE ==========
API_KEY = os.getenv("OPENAI_API_KEY", "your-api-key-here")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
# ============================================

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

In [24]:
import sounddevice as sd
import scipy.io.wavfile as wav
import numpy as np
import threading
import queue

print("✅ Simple audio recorder ready")

def record_audio(filename="recorded_audio.wav", sample_rate=16000):
    """
    Simple recorder: Press Enter to START, press Enter again to STOP.
    Works like a voice assistant - records until you're done.
    """
    audio_queue = queue.Queue()
    recording = True
    
    def audio_callback(indata, frames, time_info, status):
        if recording:
            audio_queue.put(indata.copy())
    
    print("\n🎙️ Press ENTER to start recording...")
    input()
    
    print("🔴 RECORDING... Speak now!")
    print("   Press ENTER when you're done speaking.")
    
    # Start recording in background
    stream = sd.InputStream(
        samplerate=sample_rate,
        channels=1,
        dtype='float32',
        blocksize=1024,
        callback=audio_callback
    )
    stream.start()
    
    # Wait for user to press Enter to stop
    input()
    recording = False
    stream.stop()
    stream.close()
    
    # Collect all audio from queue
    audio_chunks = []
    while not audio_queue.empty():
        audio_chunks.append(audio_queue.get())
    
    if not audio_chunks:
        print("❌ No audio recorded.")
        return None
    
    # Save audio
    audio_data = np.concatenate(audio_chunks, axis=0)
    audio_int16 = (audio_data * 32767).astype(np.int16)
    wav.write(filename, sample_rate, audio_int16)
    
    duration = len(audio_data) / sample_rate
    print(f"✅ Recording saved: {duration:.1f}s → {filename}")
    return filename

✅ Simple audio recorder ready


In [ ]:
from transformers import pipeline, AutoModelForSpeechSeq2Seq, AutoProcessor
import torch

import librosa
import numpy as np

# Load Whisper model (runs locally, no API needed)
print("🔄 Loading Whisper model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

model_id = "openai/whisper-small"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)
print(f"✅ Whisper model loaded on {device}!")

def preprocess_audio(audio_path):
    """Preprocess audio for better transcription accuracy."""
    # Load audio at 16kHz (Whisper's expected sample rate)
    audio, sr = librosa.load(audio_path, sr=16000)
    
    # Normalize audio
    audio = audio / np.max(np.abs(audio) + 1e-8)
    
    # Trim silence from beginning and end
    audio, _ = librosa.effects.trim(audio, top_db=20)
    
    return {"raw": audio, "sampling_rate": sr}

def speech_to_text(audio_path):
    """
    Transcribe audio using local Whisper model with improved accuracy.
    """
    print("🔍 Preprocessing audio...")
    audio_input = preprocess_audio(audio_path)
    
    print("🔍 Transcribing audio...")
    
    # Improved generation settings for accuracy
    result = whisper_pipe(
        audio_input,
        generate_kwargs={
            "language": "english",
            "task": "transcribe",
            "do_sample": False,  # Greedy decoding for consistency
            "num_beams": 5,  # Beam search for better accuracy
            "temperature": 0.0,  # No randomness
            "no_speech_threshold": 0.5,
            "logprob_threshold": -1.0,
            "compression_ratio_threshold": 2.4,
            "condition_on_prev_tokens": False,  # Prevents hallucinations
            "return_timestamps": False,
        }
    )
    
    transcript = result["text"].strip()
    
    print("✅ Transcription complete")
    print("📝 Text:", transcript)
    
    return transcript

🔄 Loading Whisper model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

✅ Whisper model loaded on cuda!


In [26]:
from ddgs import DDGS

def web_search(query):
    print("🔎 Searching:", query)

    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=3))

    formatted = ""

    for r in results:
        formatted += f"Title: {r['title']}\n"
        formatted += f"Snippet: {r['body']}\n"
        formatted += f"Link: {r['href']}\n\n"

    return formatted

In [27]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Use this tool for real-time data like prices, weather, news.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"]
            }
        }
    }
]



In [28]:
def run_agent(user_text):

    print("🧠 Sending to model...")

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_text}
        ],
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    # 🔍 DEBUG: Did tool call happen?
    if message.tool_calls:
        print("🔧 TOOL CALL TRIGGERED")
        print("Tool name:", message.tool_calls[0].function.name)
        print("Arguments:", message.tool_calls[0].function.arguments)

        tool_call = message.tool_calls[0]
        tool_args = json.loads(tool_call.function.arguments)

        search_results = web_search(tool_args["query"])

        # Convert message to proper dict format
        assistant_message = {
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments
                    }
                }
            ]
        }

        second_response = client.chat.completions.create(
            model="gpt-4.1-nano",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_text},
                assistant_message,
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": search_results
                }
            ]
        )

        return second_response.choices[0].message.content

    else:
        print("❌ No tool call. Direct answer.")
        return message.content

In [29]:
from IPython.display import HTML, display
import base64

def speak(text):

    print("🔊 Speaking...")

    tts_response = client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text
    )

    audio_bytes = tts_response.content
    b64 = base64.b64encode(audio_bytes).decode()

    audio_html = f"""
    <audio autoplay controls>
        <source src="data:audio/mp3;base64,{b64}" type="audio/mp3">
    </audio>
    """

    display(HTML(audio_html))

    print("✅ Done speaking.")

In [30]:
while True:
    print("\n" + "="*50)
    print("🎤 Voice Assistant")
    print("="*50)
    print("Type 'quit' to exit, or press Enter to record.")
    
    user_choice = input("> ")

    if user_choice.strip().lower() == "quit":
        print("👋 Exiting assistant.")
        break

    # 🎙 Record Audio (manual start/stop)
    audio_path = record_audio()

    if not audio_path:
        print("❌ Recording failed. Try again.")
        continue

    # 🧠 Transcribe
    print("\n🔄 Transcribing with Whisper...")
    user_text = speech_to_text(audio_path)

    if not user_text.strip():
        print("⚠️ Empty transcription. Try speaking louder.")
        continue

    print("\n🗣️ You said:", user_text)

    # 🤖 Run Agent (with tool calling)
    response = run_agent(user_text)

    print("\n🤖 Assistant:", response)

    # 🔊 Speak Response
    speak(response)


🎤 Voice Assistant
Type 'quit' to exit, or press Enter to record.

🎙️ Press ENTER to start recording...
🔴 RECORDING... Speak now!
   Press ENTER when you're done speaking.
✅ Recording saved: 4.4s → recorded_audio.wav

🔄 Transcribing with Whisper...
🔍 Preprocessing audio...


Passing `generation_config` together with generation-related arguments=({'do_sample', 'num_beams'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


🔍 Transcribing audio...
✅ Transcription complete
📝 Text: 
⚠️ Empty transcription. Try speaking louder.

🎤 Voice Assistant
Type 'quit' to exit, or press Enter to record.

🎙️ Press ENTER to start recording...
🔴 RECORDING... Speak now!
   Press ENTER when you're done speaking.
✅ Recording saved: 5.6s → recorded_audio.wav

🔄 Transcribing with Whisper...
🔍 Preprocessing audio...
🔍 Transcribing audio...
✅ Transcription complete
📝 Text: How was the India versus South Africa match that was held on yesterday?

🗣️ You said: How was the India versus South Africa match that was held on yesterday?
🧠 Sending to model...
🔧 TOOL CALL TRIGGERED
Tool name: web_search
Arguments: {"query": "India vs South Africa cricket match result yesterday"}
🔎 Searching: India vs South Africa cricket match result yesterday

🤖 Assistant: The match between India and South Africa held yesterday resulted in a victory for South Africa, who defeated India by 76 runs.
🔊 Speaking...


✅ Done speaking.

🎤 Voice Assistant
Type 'quit' to exit, or press Enter to record.

🎙️ Press ENTER to start recording...
🔴 RECORDING... Speak now!
   Press ENTER when you're done speaking.
✅ Recording saved: 5.6s → recorded_audio.wav

🔄 Transcribing with Whisper...
🔍 Preprocessing audio...
🔍 Transcribing audio...
✅ Transcription complete
📝 Text: How India can still qualify for the T20 World Cup in Super 8?

🗣️ You said: How India can still qualify for the T20 World Cup in Super 8?
🧠 Sending to model...
🔧 TOOL CALL TRIGGERED
Tool name: web_search
Arguments: {"query": "India T20 World Cup Super 8 qualification scenario"}
🔎 Searching: India T20 World Cup Super 8 qualification scenario

🤖 Assistant: Based on recent information, India can still qualify for the T20 World Cup semifinals in the Super 8 stage through the following ways:
- Improving their net run rate (NRR), which has been affected negatively after a heavy loss to South Africa.
- Winning their remaining matches in the Super 8 st

✅ Done speaking.

🎤 Voice Assistant
Type 'quit' to exit, or press Enter to record.

🎙️ Press ENTER to start recording...
🔴 RECORDING... Speak now!
   Press ENTER when you're done speaking.
✅ Recording saved: 3.4s → recorded_audio.wav

🔄 Transcribing with Whisper...
🔍 Preprocessing audio...
🔍 Transcribing audio...
✅ Transcription complete
📝 Text: Habu was the veteran Satyamangala.

🗣️ You said: Habu was the veteran Satyamangala.
🧠 Sending to model...
❌ No tool call. Direct answer.

🤖 Assistant: It appears that there might be a typo or some missing context in your statement. Could you please clarify or provide more details about "Habu" and "Satyamangala"? This will help me assist you better.
🔊 Speaking...


...

✅ Done speaking.

🎤 Voice Assistant
Type 'quit' to exit, or press Enter to record.

🎙️ Press ENTER to start recording...
🔴 RECORDING... Speak now!
   Press ENTER when you're done speaking.
✅ Recording saved: 3.9s → recorded_audio.wav

🔄 Transcribing with Whisper...
🔍 Preprocessing audio...
🔍 Transcribing audio...
✅ Transcription complete
📝 Text: How was the weather in Satyabat?

🗣️ You said: How was the weather in Satyabat?
🧠 Sending to model...
🔧 TOOL CALL TRIGGERED
Tool name: web_search
Arguments: {"query":"Weather in Satyabat"}
🔎 Searching: Weather in Satyabat

🤖 Assistant: It appears there are multiple locations with similar names. Could you please specify which Satyabat or provide the country or region so I can give you the most accurate weather information?
🔊 Speaking...


✅ Done speaking.

🎤 Voice Assistant
Type 'quit' to exit, or press Enter to record.

🎙️ Press ENTER to start recording...
🔴 RECORDING... Speak now!
   Press ENTER when you're done speaking.
✅ Recording saved: 4.0s → recorded_audio.wav

🔄 Transcribing with Whisper...
🔍 Preprocessing audio...
🔍 Transcribing audio...
✅ Transcription complete
📝 Text: How was the Veteran Satyama Kapirov?

🗣️ You said: How was the Veteran Satyama Kapirov?
🧠 Sending to model...
❌ No tool call. Direct answer.

🤖 Assistant: Could you please provide more context or specify which Veteran Satyama Kapirov you are referring to? I want to ensure I give you the most accurate information.
🔊 Speaking...


✅ Done speaking.

🎤 Voice Assistant
Type 'quit' to exit, or press Enter to record.

🎙️ Press ENTER to start recording...
🔴 RECORDING... Speak now!
   Press ENTER when you're done speaking.
✅ Recording saved: 7.5s → recorded_audio.wav

🔄 Transcribing with Whisper...
🔍 Preprocessing audio...
🔍 Transcribing audio...
✅ Transcription complete
📝 Text: How was the weather in Namakar?

🗣️ You said: How was the weather in Namakar?
🧠 Sending to model...
🔧 TOOL CALL TRIGGERED
Tool name: web_search
Arguments: {"query":"weather in Namakar"}
🔎 Searching: weather in Namakar

🤖 Assistant: I found recent weather forecasts for Namakkal, Tamil Nadu. Would you like the current weather conditions, the forecast for today, or the upcoming days?
🔊 Speaking...


✅ Done speaking.

🎤 Voice Assistant
Type 'quit' to exit, or press Enter to record.

🎙️ Press ENTER to start recording...
🔴 RECORDING... Speak now!
   Press ENTER when you're done speaking.
✅ Recording saved: 4.0s → recorded_audio.wav

🔄 Transcribing with Whisper...
🔍 Preprocessing audio...
🔍 Transcribing audio...
✅ Transcription complete
📝 Text: How was the weather in Namakalpure?

🗣️ You said: How was the weather in Namakalpure?
🧠 Sending to model...
🔧 TOOL CALL TRIGGERED
Tool name: web_search
Arguments: {"query":"Weather in Namakalpure"}
🔎 Searching: Weather in Namakalpure

🤖 Assistant: The search results indicate that there are weather forecasts available for Makalpur in Madhya Pradesh and West Bengal, but I couldn't find specific weather information for Namakalpure. If you could clarify or provide more details about the location, I can try to assist further.
🔊 Speaking...


✅ Done speaking.

🎤 Voice Assistant
Type 'quit' to exit, or press Enter to record.

🎙️ Press ENTER to start recording...
🔴 RECORDING... Speak now!
   Press ENTER when you're done speaking.
✅ Recording saved: 0.6s → recorded_audio.wav

🔄 Transcribing with Whisper...
🔍 Preprocessing audio...
🔍 Transcribing audio...
✅ Transcription complete
📝 Text: 
⚠️ Empty transcription. Try speaking louder.

🎤 Voice Assistant
Type 'quit' to exit, or press Enter to record.

🎙️ Press ENTER to start recording...
🔴 RECORDING... Speak now!
   Press ENTER when you're done speaking.
❌ No audio recorded.
❌ Recording failed. Try again.

🎤 Voice Assistant
Type 'quit' to exit, or press Enter to record.
👋 Exiting assistant.
